# Example 15 island-cycle investigation

This notebook is an audit tool for the two genuine island states that reached RAPID preparation but contain a directed graph cycle. It reads retained outputs only: the original and collapsed masks, collapse footprint, regenerated centreline, and directed network. It does not regenerate or alter any variant.

The question is deliberately narrow: is the problematic loop already present in the regenerated centreline, or was it introduced by direction assignment?

In [6]:
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import rasterio
from rasterio.windows import Window, from_bounds
from shapely.geometry import box

OUTPUT_ROOT = Path('/Volumes/PhD/river_hierarchy/network_variants/output')
EXAMPLE_ID = '15'
ISLAND_ROOT = OUTPUT_ROOT / f'{EXAMPLE_ID}_island__anchored'

def project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'rivgraph_centerline').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root containing rivgraph_centerline.')

PROJECT_ROOT = project_root()
BASE_CENTERLINE = PROJECT_ROOT / 'rivgraph_centerline' / 'outputs' / 'runs' / EXAMPLE_ID / 'rivgraph' / f'{EXAMPLE_ID}_centerline.gpkg'

# Change this value and rerun the remaining cells. The listed links are the
# directed cycles identified from the retained routing-width link layers.
ISLAND_ID = 'island_15_044'
CYCLE_LINKS = {
    'island_15_044': [118, 119, 122, 126],  # smallest of three overlapping cycles
    'island_15_111': [332, 333, 334, 340, 345, 348],
}
HALO_M = 1_000.0

if ISLAND_ID not in CYCLE_LINKS:
    raise ValueError(f'No recorded cycle links for {ISLAND_ID}.')
STATE_DIR = ISLAND_ROOT / 'island_reruns' / ISLAND_ID
VARIANT_DIR = STATE_DIR / 'variant'
CYCLE_IDS = CYCLE_LINKS[ISLAND_ID]
print('State directory:', STATE_DIR)
print('Cycle link IDs:', CYCLE_IDS)

State directory: /Volumes/PhD/river_hierarchy/network_variants/output/15_island__anchored/island_reruns/island_15_044
Cycle link IDs: [118, 119, 122, 126]


In [7]:
paths = {
    'origin_mask': VARIANT_DIR / 'mask' / 'origin_mask.tif',
    'collapsed_mask': VARIANT_DIR / 'mask' / f'{ISLAND_ID}_collapsed.tif',
    'collapse_edit': VARIANT_DIR / 'mask' / 'collapse_edit_geometries.gpkg',
    'centerline': VARIANT_DIR / 'rivgraph' / f'{ISLAND_ID}_centerline.gpkg',
    'directed_links': VARIANT_DIR / 'directed' / 'directed_links.gpkg',
    'directed_nodes': VARIANT_DIR / 'directed' / 'directed_nodes.gpkg',
    'base_centerline': BASE_CENTERLINE,
}
missing = [name for name, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing retained review products: {missing}')

links = gpd.read_file(paths['directed_links'])
base_centerline = gpd.read_file(paths['base_centerline']).to_crs(links.crs)
nodes = gpd.read_file(paths['directed_nodes'])
edits = gpd.read_file(paths['collapse_edit'])
cycle_links = links.loc[links['id_link'].astype(int).isin(CYCLE_IDS)].copy()
if len(cycle_links) != len(CYCLE_IDS):
    raise ValueError(f'Expected {len(CYCLE_IDS)} cycle links but found {len(cycle_links)}.')

graph = nx.DiGraph()
for row in links[['id_link', 'id_us_node', 'id_ds_node']].itertuples(index=False):
    graph.add_edge(int(row.id_us_node), int(row.id_ds_node), id_link=int(row.id_link))
cycles = list(nx.simple_cycles(graph))

def cycle_link_ids(cycle):
    return [graph[a][b]['id_link'] for a, b in zip(cycle, cycle[1:] + cycle[:1])]

cycle_summary = pd.DataFrame([
    {'cycle_nodes': cycle, 'cycle_link_ids': cycle_link_ids(cycle), 'n_links': len(cycle)}
    for cycle in cycles
])
display(cycle_summary)
display(cycle_links[['id_link', 'id_us_node', 'id_ds_node', 'geometry']].sort_values('id_link'))

,cycle_nodes,cycle_link_ids,n_links
0,"[67, 79, 78, 75, 77, 80, 74, 76, 83, 87, 84]","[106, 124, 121, 120, 123, 119, 118, 122, 129, ...",11
1,"[74, 76, 83, 80]","[118, 122, 126, 119]",4
2,"[74, 76, 83, 87, 84, 79, 78, 75, 77, 80]","[118, 122, 129, 130, 125, 124, 121, 120, 123, ...",10


,id_link,id_us_node,id_ds_node,geometry
118,118,74,76,"LINESTRING (786305.148 515705.814, 786295.204 ..."
119,119,80,74,"LINESTRING (786325.037 515666.036, 786325.037 ..."
122,122,76,83,"LINESTRING (785947.152 515685.925, 785947.152 ..."
126,126,83,80,"LINESTRING (786285.26 515556.649, 786285.26 51..."


In [8]:
# Build an oriented, local display. Geometry is reversed where necessary so
# each displayed line runs from id_us_node to id_ds_node.
node_geometry = nodes.set_index('id_node').geometry.to_dict()

def line_coordinates_oriented(row):
    geometry = row.geometry
    if geometry.geom_type == 'MultiLineString':
        geometry = max(geometry.geoms, key=lambda part: part.length)
    coordinates = np.asarray(geometry.coords)
    upstream = node_geometry[int(row.id_us_node)]
    downstream = node_geometry[int(row.id_ds_node)]
    forward_error = geometry.boundary.geoms[0].distance(upstream) + geometry.boundary.geoms[-1].distance(downstream)
    reverse_error = geometry.boundary.geoms[-1].distance(upstream) + geometry.boundary.geoms[0].distance(downstream)
    return coordinates if forward_error <= reverse_error else coordinates[::-1]

cycle_bounds = cycle_links.total_bounds
minx, miny, maxx, maxy = (cycle_bounds[0] - HALO_M, cycle_bounds[1] - HALO_M,
                         cycle_bounds[2] + HALO_M, cycle_bounds[3] + HALO_M)
review_window_geometry = box(minx, miny, maxx, maxy)
local_links = links.loc[links.intersects(review_window_geometry)].copy()
base_local_centerline = base_centerline.loc[base_centerline.intersects(review_window_geometry)].copy()
local_edits = edits.loc[edits.intersects(review_window_geometry)].copy()
print(f'All directed links shown around the cycle: {len(local_links)}')
print(f'Original base-centreline segments shown: {len(base_local_centerline)}')

All directed links shown around the cycle: 64
Original base-centreline segments shown: 1


In [9]:
with rasterio.open(paths['collapsed_mask']) as source:
    transform = source.transform
    bounds = source.bounds
    window = from_bounds(minx, miny, maxx, maxy, transform=transform)
    window = window.round_offsets().round_lengths()
    window = window.intersection(Window(0, 0, source.width, source.height))
    collapsed = source.read(1, window=window).astype(bool)
with rasterio.open(paths['origin_mask']) as source:
    origin = source.read(1, window=window).astype(bool)

# Blue = water retained/created in the collapsed mask; orange = pixels made wet
# by this island collapse. White = dry land.
image = np.full((*collapsed.shape, 3), 248, dtype=np.uint8)
image[collapsed] = (80, 130, 180)
image[(~origin) & collapsed] = (245, 133, 24)
inverse_transform = ~transform

def to_local_pixels(coordinates):
    cols, rows = inverse_transform * (coordinates[:, 0], coordinates[:, 1])
    return np.asarray(cols) - window.col_off, np.asarray(rows) - window.row_off

def geometry_pixel_lines(frame):
    line_x, line_y = [], []
    for geometry in frame.geometry:
        if geometry is None or geometry.is_empty:
            continue
        if geometry.geom_type in {'Polygon', 'MultiPolygon'}:
            geometry = geometry.boundary
        parts = geometry.geoms if hasattr(geometry, 'geoms') else [geometry]
        for part in parts:
            coordinates = np.asarray(part.coords)
            xs, ys = to_local_pixels(coordinates)
            line_x.extend(xs.tolist() + [None])
            line_y.extend(ys.tolist() + [None])
    return line_x, line_y

background_x, background_y = [], []
for row in local_links.itertuples(index=False):
    coords = line_coordinates_oriented(row)
    xs, ys = to_local_pixels(coords)
    background_x.extend(xs.tolist() + [None])
    background_y.extend(ys.tolist() + [None])

base_x, base_y = geometry_pixel_lines(base_local_centerline)
edit_x, edit_y = geometry_pixel_lines(local_edits)
fig = go.Figure(go.Image(z=image))
fig.add_trace(go.Scatter(x=base_x, y=base_y, mode='lines', name='Original independent base centreline',
    line={'color': 'rgba(70,70,70,0.85)', 'width': 2, 'dash': 'dash'}, hoverinfo='skip'))
fig.add_trace(go.Scattergl(x=background_x, y=background_y, mode='lines',
    name='Other directed links', line={'color': 'rgba(0,0,0,0.35)', 'width': 1}, hoverinfo='skip'))
if edit_x:
    fig.add_trace(go.Scatter(x=edit_x, y=edit_y, mode='lines', name='Collapsed-island footprint',
        line={'color': '#6A3D9A', 'width': 3}, hoverinfo='skip'))

for row in cycle_links.sort_values('id_link').itertuples(index=False):
    coords = line_coordinates_oriented(row)
    xs, ys = to_local_pixels(coords)
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines+markers', name=f'Cycle link {int(row.id_link)}: {int(row.id_us_node)} → {int(row.id_ds_node)}',
        line={'color': '#D62728', 'width': 4}, marker={'size': [0] * (len(xs)-1) + [8], 'color': '#D62728'},
        hovertemplate='Link ' + str(int(row.id_link)) + '<br>upstream node ' + str(int(row.id_us_node)) + '<br>downstream node ' + str(int(row.id_ds_node)) + '<extra></extra>'
    ))

cycle_node_ids = sorted(set(cycle_links['id_us_node'].astype(int)) | set(cycle_links['id_ds_node'].astype(int)))
cycle_nodes = nodes.loc[nodes['id_node'].astype(int).isin(cycle_node_ids)]
node_xy = np.asarray([(point.x, point.y) for point in cycle_nodes.geometry])
node_x, node_y = to_local_pixels(node_xy)
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text', name='Cycle nodes',
    text=cycle_nodes['id_node'].astype(int), textposition='top center',
    marker={'size': 9, 'color': '#111111'}, hovertemplate='Node %{text}<extra></extra>'))
fig.update_layout(
    title=f'{ISLAND_ID}: red lines form the directed cycle; orange/purple show the collapsed island; dashed grey is the original base centreline',
    template='plotly_white', width=1100, height=850, margin={'l': 10, 'r': 10, 't': 55, 'b': 10}
)
fig.update_xaxes(visible=False, scaleanchor='y', scaleratio=1)
fig.update_yaxes(visible=False, autorange='reversed')
fig.show()

## Interpretation and QGIS hand-off

The red lines are explicitly oriented from `id_us_node` to `id_ds_node`; the final marker marks the downstream end. Orange pixels and the purple outline show water added by this island collapse. The dashed grey line is the original independent-base centreline. If the red links form a loop while both the black regenerated centreline and dashed base centreline are locally tree-like, direction assignment is the likely issue. If the black regenerated centreline itself contains the loop, it is a RivGraph/regeneration topology issue.

For a QGIS check, open the printed state directory's `mask/*collapsed.tif`, `mask/collapse_edit_geometries.gpkg`, `rivgraph/*centerline.gpkg`, and `directed/directed_links.gpkg`. Filter the directed-link layer on the currently selected `CYCLE_LINKS[ISLAND_ID]`, and symbolize it with arrowheads. Do not edit these retained audit products.